# 🏦 Microfinance Loan Delinquency EWS — Exploration & Modeling

This notebook covers the design, exploration, and training of the Early Warning System (EWS) prototype for predicting borrower delinquency using pre-loan income patterns.

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
data_path = Path("../data/raw/farmers_salary_transactions.csv")
df_raw = pd.read_csv(data_path)
print(f"Raw Data Shape: {df_raw.shape}")
df_raw.head()

## 2. Exploratory Data Analysis (EDA)
Let's check the distribution of average incomes and visualize some transaction histories.

In [ ]:
# Pre-loan lookback features are computed from weeks 1-8
week_cols = [f"week_{i}" for i in range(1, 9)]
df_lookback = df_raw[week_cols]

# Describe pre-loan lookback incomes
df_lookback.mean(axis=1).describe()

In [ ]:
# Plot the average pre-loan weekly income distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_lookback.mean(axis=1), bins=30, kde=True, color="teal")
plt.title("Distribution of Average Pre-Loan Weekly Income (Weeks 1-8)")
plt.xlabel("Average Weekly Income (₹)")
plt.ylabel("Count")
plt.show()

## 3. Modeling and Evaluation
We load the processed dataset `labeled_borrowers.csv` to evaluate our classification performance.

In [ ]:
proc_path = Path("../data/processed/labeled_borrowers.csv")
if proc_path.exists():
    df_proc = pd.read_csv(proc_path)
    print(f"Processed Data Shape: {df_proc.shape}")
    print(df_proc["delinquent"].value_counts(normalize=True))
else:
    print("Processed data not found! Run run_pipeline.py first.")

## 4. Key Takeaways & Observations
- **Data Leakage Guard**: Pre-loan features are strictly bounded within weeks 1-8, while loan repayment labels are computed from weeks 9-28.
- **XGBoost Performance**: The tree-based XGBoost model captures non-linear risk trends and volatility interactions much better than simple linear models.
- **Cost-Sensitive Threshold**: Lowering the classification threshold to `0.05` drastically reduces the Total Cost of defaults, preserving microfinance capital.